In [1]:
import csv
import os
import time
import requests
import re
import psycopg2
from psycopg2 import sql
from datetime import datetime, timezone
from typing import Optional

import sys
sys.path.append("..")
from match_url_checker import *
from db_operations import get_db_conn
from dotenv import load_dotenv
load_dotenv("../../src/config/.env")

API_CALL_DELAY_SECONDS = 1.2

In [8]:
def import_csv_to_table(csv_file, table_name, drop_if_exists=True):
    """Read CSV and insert data into PostgreSQL table."""
    conn = get_db_conn()
    cur = conn.cursor()

    # Drop table if needed
    if drop_if_exists:
        cur.execute(sql.SQL("DROP TABLE IF EXISTS {}").format(sql.Identifier(table_name)))

    # Create table
    create_table_query = sql.SQL("""
        CREATE TABLE IF NOT EXISTS {table} (
            hash TEXT,
            project_id BIGINT,
            version BIGINT,
            license TEXT,
            method_name TEXT,
            file_location TEXT,
            repository_url TEXT,
            language TEXT,
            granular_level_reached INT,
            raw_url TEXT,
            line_content TEXT,
            target_line_number TEXT
        )
    """).format(table=sql.Identifier(table_name))
    cur.execute(create_table_query)

    # Read CSV and insert
    with open(csv_file, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        header += ['granular_level_reached', 'raw_url', 'line_content', 'target_line_number']
        
        insert_columns = [
            "hash", "project_id", "version", "license", "method_name",
            "file_location", "repository_url", "language",
            "granular_level_reached", "raw_url", "line_content", "target_line_number"
        ]
        placeholders = sql.SQL(", ").join([sql.Placeholder() for _ in insert_columns])
        insert_query = sql.SQL("""
            INSERT INTO {table} ({fields})
            VALUES ({values})
        """).format(
            table=sql.Identifier(table_name),
            fields=sql.SQL(", ").join(map(sql.Identifier, insert_columns)),
            values=placeholders
        )

        for row in reader:
            cur.execute(insert_query, row + [0, "", "", ""])  # default placeholders

    conn.commit()
    cur.close()
    conn.close()
    print(f"CSV imported into table '{table_name}' successfully.")


In [9]:
def update_granular_levels(TABLE_NAME):
    conn = get_db_conn()
    cur = conn.cursor()

    # --- Fetch rows from table ---
    cur.execute(f"SELECT hash, project_id, version, license, method_name, file_location, repository_url, language FROM {TABLE_NAME}")
    rows = cur.fetchall()

    for i, row in enumerate(rows, 1):
        try:
            print(f"Processing row {i}...")
            hash_val, project_id, version, license_, method_name, file_location, repo_url, language = row

            # Default values
            granular_level_reached = 0
            raw_url = ""
            line_content = ""
            target_line_number = ""

            # Verification steps
            if github_repo_exists(repo_url):
                granular_level_reached = 1
                commit_sha, _, _ = check_version_and_get_sha(repo_url, version)
                if commit_sha:
                    granular_level_reached = 2
                    if find_file_at_exact_path(repo_url):
                        granular_level_reached = 3
                        if find_method_in_file(repo_url, method_name):
                            granular_level_reached = 4

            # --- Update DB row ---
            update_query = f"""
                UPDATE {TABLE_NAME}
                SET granular_level_reached = %s,
                    raw_url = %s,
                    line_content = %s,
                    target_line_number = %s
                WHERE hash = %s AND project_id = %s AND version = %s
            """
            cur.execute(update_query, (granular_level_reached, raw_url, line_content, target_line_number,
                                       hash_val, project_id, version))
        except Exception as e:
            print(f"Error processing row {i}: {e}")
            continue

    conn.commit()
    cur.close()
    conn.close()
    print("Granular levels updated for all rows.")

In [10]:
INPUT_CSV_FILE = '../results/data/verifier_demo.csv'
TABLE_NAME = "repository_data_source_verifier"  # change if needed
DROP_IF_EXISTS = True

def main():
    import_csv_to_table('../results/data/verifier_demo.csv', TABLE_NAME, DROP_IF_EXISTS)
    update_granular_levels(TABLE_NAME)

In [11]:
if __name__ == '__main__':
    main()

CSV imported into table 'repository_data_source_verifier' successfully.
Processing row 1...
--> Found version for timestamp 1756905068000 in IBM/terratorch. Commit SHA: 821e80b6407af2cb4eb70ccc689b53915a1fde45
Processing row 2...
--> Found version for timestamp 1756894106000 in IBM/unitxt. Commit SHA: fa5b604db4c49142adc276879be90388840b8d24
Processing row 3...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 4...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 5...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 6...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 7...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f

In [ ]:
'''
\copy (SELECT * FROM repository_data_source_verifier LIMIT 10) TO '/tmp/validation_mined_data.csv' WITH CSV HEADER;

sudo mv /tmp/validation_mined_data.csv /searchSECO-miner/results/data
'''

''

In [2]:
import pandas as pd
from tqdm import tqdm  # optional, shows progress bar

def enrich_csv_with_granular_level(csv_file: str, output_file: str):
    """
    Reads a CSV, checks GitHub repo/method information, and adds a 
    'granular_level_reached' column to the CSV.
    
    Parameters:
        csv_file (str): Path to input CSV.
        output_file (str): Path to save enriched CSV.
    """
    # Read CSV
    df = pd.read_csv(csv_file, encoding='utf-8')
    
    # Ensure required columns exist
    required_cols = ['repository_url', 'version', 'method_name']
    for col in required_cols:
        if col not in df.columns:
            df[col] = None  # fill missing with None
    
    # Initialize granular_level column
    df['granular_level_reached'] = 0

    # Iterate through rows
    for idx, row in df.iterrows():
        try:
            repo_url = row['repository_url']
            version = row['version']
            method_name = row['method_name']
            level = 0

            if github_repo_exists(repo_url):
                level = 1
                commit_sha, _, _ = check_version_and_get_sha(repo_url, version)
                if commit_sha:
                    level = 2
                    if find_file_at_exact_path(repo_url):
                        level = 3
                        if find_method_in_file(repo_url, method_name):
                            level = 4

            df.at[idx, 'granular_level_reached'] = level
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    
    # Save enriched CSV
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Enriched CSV saved to {output_file}")


In [3]:
OUTPUT_CSV = "../../results/processed_data/Samsung_mTower_matches_vr.csv"
INPUT_CSV_FILE = "../../results/processed_data/Samsung_mTower_matches.csv"

enrich_csv_with_granular_level(INPUT_CSV_FILE, OUTPUT_CSV)

--> Found version for timestamp 1673494326000 in NVIDIA/optee_os-nvidia. Commit SHA: cb9fb78e9ac480daa906b1689954f0a4173e9906
--> Found version for timestamp 1747392693000 in Samsung/mTower. Commit SHA: adf9d8ffcf06af4ee3386d6d1ac384c47e9cd0b0
--> Found version for timestamp 1702987255000 in sifive/freedom-metal. Commit SHA: fa026d2ee08e5ba49e8ae703fb4cbcbb710a6a69
--> Found version for timestamp 1747392693000 in Samsung/mTower. Commit SHA: adf9d8ffcf06af4ee3386d6d1ac384c47e9cd0b0
--> Found version for timestamp 1734335548000 in bouffalolab/bl_iot_sdk. Commit SHA: edcf613f073d3ef7d2f290d7c285b69be35da75b
--> Found version for timestamp 1747392693000 in Samsung/mTower. Commit SHA: adf9d8ffcf06af4ee3386d6d1ac384c47e9cd0b0
--> Found version for timestamp 1702987255000 in sifive/freedom-metal. Commit SHA: fa026d2ee08e5ba49e8ae703fb4cbcbb710a6a69
--> Found version for timestamp 1747392693000 in Samsung/mTower. Commit SHA: adf9d8ffcf06af4ee3386d6d1ac384c47e9cd0b0
--> Found version for timest

In [4]:
from method_extractor import process_code_extraction

process_code_extraction("../../results/processed_data/Samsung_mTower_matches_vr.csv", "../../results/processed_data/Samsung_mTower_matches_extr.csv")

Starting the processing of ../../results/processed_data/Samsung_mTower_matches_vr.csv...
Loaded 2230 rows from CSV
Found 2158 entries with granular level 4
Processing row 2: https://github.com/NVIDIA/optee_os-nvidia/blob/cb9fb78e9ac480daa906b1689954f0a4173e9906/./core/lib/libtomcrypt/hash.c#L172
  -> Successfully extracted method
Processing row 3: https://github.com/Samsung/mTower/blob/master/crypto/libtomcrypt/src/tee_ltc_provider.c#L3076
  -> Successfully extracted method
Processing row 4: https://github.com/sifive/freedom-metal/blob/fa026d2ee08e5ba49e8ae703fb4cbcbb710a6a69/./src/drivers/riscv_clint0.c#L210
  -> Successfully extracted method
Processing row 5: https://github.com/Samsung/mTower/blob/master/arch/riscv32/fe310/src/freedom-metal/src/drivers/riscv_clint0.c#L210
  -> Successfully extracted method
Processing row 6: https://github.com/bouffalolab/bl_iot_sdk/blob/edcf613f073d3ef7d2f290d7c285b69be35da75b/./components/platform/soc/bl808/bl808_e907_std/bl808_bsp_driver/std_drv/sr

In [ ]:
def compare_clone_pairs(df, parsers):
    """Compares adjacent rows (source/clone pairs) with hash optimization."""
    
    for i in range(0, len(df), 2):
        if i + 1 < len(df):
            source_row = df.iloc[i]
            clone_row = df.iloc[i+1]
            
            # Fast path: check if hashes are identical (exact match)
            if pd.notna(source_row['upi_hash']) and pd.notna(clone_row['upi_hash']):
                if source_row['upi_hash'] == clone_row['upi_hash']:
                    similarity_score = 1.0
                else:
                    # Compute actual similarity only if hashes differ
                    similarity_score = upi_similarity(source_row['upi_seq'], clone_row['upi_seq'])
            else:
                # Fallback: compute without hash optimization
                lang = source_row['language']
                parser = parsers.get(lang)
                if not parser:
                    continue
                
                tree_src = parser.parse(bytes(source_row['function_code'], 'utf8'))
                upi_seq_src = ast_to_upi_sequence(tree_src.root_node)
                
                tree_clone = parser.parse(bytes(clone_row['function_code'], 'utf8'))
                upi_seq_clone = ast_to_upi_sequence(tree_clone.root_node)
                
                similarity_score = upi_similarity(upi_seq_src, upi_seq_clone)
            
            # Update the DataFrame for the clone row
            df.at[df.index[i+1], 'similarity_score'] = similarity_score
            
    
    return df